# 04. Motor conversacional RAG — versión final

Notebook final de la Fase 4 del TFM.


Configuración final:

- embeddings multilingües `paraphrase-multilingual-MiniLM-L12-v2`;
- chunking estructurado;
- ChromaDB local;
- retrieval híbrido v2.2;
- búsqueda específica cuando se menciona un documento;
- expansión de contexto para modificaciones normativas;
- mejora específica de consultas sobre impacto residencial;
- prompt v3;
- modelo generativo final: `mistral:7b`;
- evaluación final sobre 8 preguntas representativas.

Las secciones experimentales de comparación entre Llama y Mistral se han eliminado
de esta versión final.

## 1. Configuración

Se definen rutas, parámetros de chunking, configuración del retrieval,
modelo de embeddings y modelos locales.

Esta versión final puede ejecutarse secuencialmente de arriba abajo.

In [ ]:
from pathlib import Path
import os
import re
import json
import hashlib
import time
import warnings
import unicodedata

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import chromadb

warnings.filterwarnings("ignore")

RAG_DIR = Path("../data/RAG")

PROCESSED_RAG_DIR = Path(
    "../data/processed/rag_v2"
)

VECTOR_DB_DIR = Path(
    "../data/vectorstore/chroma_vut_malaga_v2"
)

REPORTS_RAG_DIR = Path(
    "../reports/rag_v2"
)

for directory in [
    PROCESSED_RAG_DIR,
    VECTOR_DB_DIR,
    REPORTS_RAG_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

EMBEDDING_MODEL_NAME = (
    "paraphrase-multilingual-MiniLM-L12-v2"
)

COLLECTION_NAME = (
    "vut_malaga_normativa_v2"
)

# Retrieval
CANDIDATE_POOL = 15
FINAL_TOP_K = 4
MIN_SIMILARITY = 0.28
NEAR_DUPLICATE_THRESHOLD = 0.93
MMR_LAMBDA = 0.72
MAX_PER_DOCUMENT = 2

# Chunking
TARGET_CHUNK_WORDS = 190
MAX_CHUNK_WORDS = 260
MIN_CHUNK_WORDS = 35

# Modelo seleccionado tras la comparación experimental
BASELINE_LLM = "llama3.2:3b"
CANDIDATE_LLM = "mistral:7b"
FINAL_LLM = "mistral:7b"

print("Configuración RAG v2 cargada")
print("Embeddings:", EMBEDDING_MODEL_NAME)
print("Colección:", COLLECTION_NAME)
print("LLM final:", FINAL_LLM)


## 2. Inventario y metadatos documentales

Se conserva la distinción entre normativa autonómica, instrumentos municipales, informes y planeamiento.

La detección del **informe jurídico** se realiza antes que la de `Instrucción 1/2024`, porque su nombre de archivo puede contener también la palabra `instruccion`.


In [ ]:
def infer_document_metadata(filename):
    name = filename.lower()

    meta = {
        "document_type": "documento_tecnico",
        "administration": "Ayuntamiento de Málaga",
        "scope": "Málaga",
        "short_title": Path(filename).stem,
    }

    if "decreto_31" in name:
        meta.update(
            document_type="normativa_autonomica",
            administration="Junta de Andalucía",
            scope="Andalucía",
            short_title="Decreto 31/2024",
        )

    elif (
        "decreto_texto" in name
        or "28_2016" in name
    ):
        meta.update(
            document_type="normativa_autonomica",
            administration="Junta de Andalucía",
            scope="Andalucía",
            short_title="Decreto 28/2016",
        )

    # Debe comprobarse antes de "instruccion".
    elif "informe-juridico" in name:
        meta.update(
            document_type="informe_juridico",
            short_title=(
                "Informe jurídico sobre Instrucción 1/2024"
            ),
        )

    elif "instruccion" in name:
        meta.update(
            document_type="instruccion_municipal",
            short_title="Instrucción 1/2024",
        )

    elif "resolucion" in name:
        meta.update(
            document_type="resolucion_municipal",
            short_title=(
                "Resolución sobre Instrucción 1/2024"
            ),
        )

    elif "resumen_ejecutivo" in name:
        meta.update(
            document_type="planeamiento_urbanistico",
            short_title="Resumen Ejecutivo PGOU VUT",
        )

    elif "memoria" in name:
        meta.update(
            document_type="planeamiento_urbanistico",
            short_title="Memoria modificación PGOU VUT",
        )

    elif "acuerdo-pleno" in name:
        meta.update(
            document_type="acuerdo_municipal",
            short_title=(
                "Acuerdo Pleno aprobación definitiva"
            ),
        )

    elif "impacto" in name:
        meta.update(
            document_type="informe_tecnico",
            short_title=(
                "Informe de Impacto de la Vivienda Turística"
            ),
        )

    elif "plano" in name:
        meta.update(
            document_type="cartografia",
            short_title=(
                "Plano de viviendas turísticas por barrios"
            ),
        )

    return meta


pdf_files = sorted(
    RAG_DIR.glob("*.pdf")
)

if not pdf_files:
    raise FileNotFoundError(
        f"No se han encontrado PDF en {RAG_DIR.resolve()}"
    )

document_inventory = pd.DataFrame(
    [
        {
            "archivo": pdf_path.name,
            **infer_document_metadata(
                pdf_path.name
            ),
        }
        for pdf_path in pdf_files
    ]
)

print("PDF encontrados:", len(pdf_files))
display(document_inventory)


## 3. Extracción y limpieza

En esta versión se intenta conservar la estructura de líneas para detectar mejor encabezados, artículos y apartados.

Se eliminan firmas, códigos de verificación, URLs administrativas y otras líneas repetitivas.


In [ ]:
NOISE_PATTERNS = [
    r"^Código Seguro De Verificación.*$",
    r"^Codigo Seguro De Verificacion.*$",
    r"^Firmado Por.*$",
    r"^Observaciones Página.*$",
    r"^Observaciones Pagina.*$",
    r"^Url De Verificación.*$",
    r"^Url De Verificacion.*$",
    r"^Normativa Este informe tiene carácter.*$",
    r"^Normativa Este informe tiene caracter.*$",
    r"^Este documento no tiene validez jurídica\.?$",
    r"^Este documento no tiene validez juridica\.?$",
    r"^Boletín Oficial de la Junta de Andalucía$",
    r"^Boletin Oficial de la Junta de Andalucia$",
    r"^\d+\s*/\s*\d+$",
]

COMPILED_NOISE_PATTERNS = [
    re.compile(
        pattern,
        flags=re.IGNORECASE,
    )
    for pattern in NOISE_PATTERNS
]

EMBEDDED_NOISE_TERMS = [
    "firmado por",
    "código seguro de verificación",
    "codigo seguro de verificacion",
    "url de verificación",
    "url de verificacion",
    "valida.malaga.eu/verifirma",
    "este documento no tiene validez jurídica",
    "este documento no tiene validez juridica",
]


def is_noise_line(line):
    normalized = re.sub(
        r"\s+",
        " ",
        str(line),
    ).strip()

    if not normalized:
        return True

    lower_line = normalized.lower()

    if any(
        term in lower_line
        for term in EMBEDDED_NOISE_TERMS
    ):
        return True

    return any(
        pattern.match(normalized)
        for pattern in COMPILED_NOISE_PATTERNS
    )


def fix_split_hyphenation(text):
    return re.sub(
        r"([A-Za-zÁÉÍÓÚÜÑáéíóúüñ])-\s*\n\s*"
        r"([A-Za-zÁÉÍÓÚÜÑáéíóúüñ])",
        r"\1\2",
        text,
    )


def clean_pdf_text_preserve_lines(text):
    if not text:
        return ""

    text = text.replace(
        "\x00",
        " ",
    )

    text = fix_split_hyphenation(
        text
    )

    cleaned_lines = []

    for raw_line in text.splitlines():
        line = re.sub(
            r"\s+",
            " ",
            raw_line,
        ).strip()

        if is_noise_line(line):
            continue

        cleaned_lines.append(line)

    # Se conserva el salto de línea para la fase estructural.
    cleaned = "\n".join(
        cleaned_lines
    )

    cleaned = re.sub(
        r"\n{3,}",
        "\n\n",
        cleaned,
    )

    return cleaned.strip()


In [ ]:
page_records = []

for pdf_path in tqdm(
    pdf_files,
    desc="Extrayendo PDF",
):
    meta = infer_document_metadata(
        pdf_path.name
    )

    reader = PdfReader(
        str(pdf_path)
    )

    for page_number, page in enumerate(
        reader.pages,
        start=1,
    ):
        try:
            raw_text = (
                page.extract_text()
                or ""
            )
        except Exception:
            raw_text = ""

        clean_text = (
            clean_pdf_text_preserve_lines(
                raw_text
            )
        )

        page_records.append(
            {
                "source_file": pdf_path.name,
                **meta,
                "page": page_number,
                "raw_text": raw_text,
                "text": clean_text,
                "raw_n_chars": len(raw_text),
                "n_chars": len(clean_text),
                "n_words": len(
                    clean_text.split()
                ),
            }
        )

pages_df = pd.DataFrame(
    page_records
)

PAGES_OUTPUT_PATH = (
    PROCESSED_RAG_DIR
    / "rag_pages_v2.parquet"
)

pages_df.to_parquet(
    PAGES_OUTPUT_PATH,
    index=False,
)

print(
    "Páginas extraídas:",
    len(pages_df),
)

display(
    pages_df[
        [
            "short_title",
            "page",
            "n_words",
        ]
    ].head()
)


## 4. Segmentación estructurada y chunking

La estrategia intenta detectar primero límites normativos o administrativos:

- `Artículo ...`
- `Primero.`, `Segundo.`, etc.
- apartados numerados;
- encabezados breves en mayúsculas;
- párrafos separados.

Después, únicamente cuando un bloque es demasiado largo, se subdivide por frases.

Esto reduce la probabilidad de mezclar en un mismo chunk artículos o apartados que tratan cuestiones distintas.


In [ ]:
STRUCTURAL_PATTERNS = [
    re.compile(
        r"^art[íi]culo\s+\d+[^\n]*",
        flags=re.IGNORECASE,
    ),
    re.compile(
        r"^(primero|segundo|tercero|cuarto|quinto|sexto|"
        r"séptimo|septimo|octavo|noveno|décimo|decimo)\b",
        flags=re.IGNORECASE,
    ),
    re.compile(
        r"^\d+(?:\.\d+)*[\.\)]\s+\S+",
        flags=re.IGNORECASE,
    ),
]


def is_structural_heading(line):
    line = line.strip()

    if not line:
        return False

    if any(
        pattern.match(line)
        for pattern in STRUCTURAL_PATTERNS
    ):
        return True

    words = line.split()

    if (
        2 <= len(words) <= 12
        and len(line) <= 120
        and line.upper() == line
        and any(
            char.isalpha()
            for char in line
        )
    ):
        return True

    return False


def split_sentences(text):
    text = re.sub(
        r"\s+",
        " ",
        text,
    ).strip()

    if not text:
        return []

    parts = re.split(
        r"(?<=[.!?;:])\s+(?=[A-ZÁÉÍÓÚÜÑ0-9«“])",
        text,
    )

    return [
        part.strip()
        for part in parts
        if part.strip()
    ]


def split_into_structural_blocks(text):
    lines = [
        line.strip()
        for line in text.splitlines()
        if line.strip()
    ]

    blocks = []
    current = []

    for line in lines:
        if (
            current
            and is_structural_heading(line)
        ):
            blocks.append(
                " ".join(current).strip()
            )
            current = [line]
        else:
            current.append(line)

    if current:
        blocks.append(
            " ".join(current).strip()
        )

    return [
        block
        for block in blocks
        if block
    ]


def split_long_block(
    text,
    target_words=TARGET_CHUNK_WORDS,
    max_words=MAX_CHUNK_WORDS,
):
    words = text.split()

    if len(words) <= max_words:
        return [text.strip()]

    sentences = split_sentences(
        text
    )

    chunks = []
    current = []
    current_words = 0

    for sentence in sentences:
        n_words = len(
            sentence.split()
        )

        if (
            current
            and current_words + n_words
            > target_words
        ):
            chunks.append(
                " ".join(current).strip()
            )
            current = []
            current_words = 0

        current.append(sentence)
        current_words += n_words

    if current:
        chunks.append(
            " ".join(current).strip()
        )

    # Fallback para textos sin puntuación útil.
    safe_chunks = []

    for chunk in chunks:
        chunk_words = chunk.split()

        if len(chunk_words) <= max_words:
            safe_chunks.append(chunk)
            continue

        for start in range(
            0,
            len(chunk_words),
            target_words,
        ):
            safe_chunks.append(
                " ".join(
                    chunk_words[
                        start:
                        start + target_words
                    ]
                )
            )

    return safe_chunks


def build_page_chunks(text):
    structural_blocks = (
        split_into_structural_blocks(
            text
        )
    )

    chunks = []

    for block in structural_blocks:
        chunks.extend(
            split_long_block(
                block
            )
        )

    return [
        chunk.strip()
        for chunk in chunks
        if (
            len(chunk.split())
            >= MIN_CHUNK_WORDS
        )
    ]


In [ ]:
chunk_records = []

for _, row in pages_df.iterrows():

    if row["n_words"] < MIN_CHUNK_WORDS:
        continue

    page_chunks = build_page_chunks(
        row["text"]
    )

    for chunk_index, chunk_text in enumerate(
        page_chunks
    ):
        raw_id = (
            f"{row['source_file']}|"
            f"{row['page']}|"
            f"{chunk_index}|v2"
        )

        chunk_id = hashlib.sha1(
            raw_id.encode("utf-8")
        ).hexdigest()

        chunk_records.append(
            {
                "chunk_id": chunk_id,
                "source_file": row["source_file"],
                "short_title": row["short_title"],
                "document_type": row["document_type"],
                "administration": row["administration"],
                "scope": row["scope"],
                "page": int(row["page"]),
                "chunk_index": int(chunk_index),
                "text": chunk_text,
                "n_words": len(
                    chunk_text.split()
                ),
            }
        )

chunks_df = pd.DataFrame(
    chunk_records
)

CHUNKS_OUTPUT_PATH = (
    PROCESSED_RAG_DIR
    / "rag_chunks_v2.parquet"
)

chunks_df.to_parquet(
    CHUNKS_OUTPUT_PATH,
    index=False,
)

print(
    "Chunks finales:",
    len(chunks_df),
)

print(
    "Media palabras:",
    round(
        chunks_df["n_words"].mean(),
        1,
    ),
)

print(
    "Mediana palabras:",
    round(
        chunks_df["n_words"].median(),
        1,
    ),
)

display(
    chunks_df[
        [
            "short_title",
            "page",
            "chunk_index",
            "n_words",
            "text",
        ]
    ].head(10)
)


### 4.1. Auditoría del chunking

Antes de crear embeddings conviene inspeccionar:

- distribución de tamaños;
- fragmentos muy cortos/largos;
- posibles duplicados exactos.


In [ ]:
chunk_audit = pd.DataFrame(
    {
        "metrica": [
            "n_chunks",
            "min_words",
            "p25_words",
            "median_words",
            "mean_words",
            "p75_words",
            "max_words",
            "exact_duplicates",
        ],
        "valor": [
            len(chunks_df),
            int(chunks_df["n_words"].min()),
            float(chunks_df["n_words"].quantile(0.25)),
            float(chunks_df["n_words"].median()),
            float(chunks_df["n_words"].mean()),
            float(chunks_df["n_words"].quantile(0.75)),
            int(chunks_df["n_words"].max()),
            int(
                chunks_df["text"]
                .str.lower()
                .str.replace(
                    r"\s+",
                    " ",
                    regex=True,
                )
                .duplicated()
                .sum()
            ),
        ],
    }
)

display(chunk_audit)

display(
    chunks_df.nlargest(
        10,
        "n_words",
    )[
        [
            "short_title",
            "page",
            "n_words",
            "text",
        ]
    ]
)


## 5. Embeddings e índice ChromaDB v2

Se mantiene el modelo multilingüe anterior para aislar el efecto de las mejoras de retrieval.

Se usa **distancia coseno**, lo que permite interpretar:

```text
similarity = 1 - cosine_distance
```


In [ ]:
embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME
)

embeddings = embedding_model.encode(
    chunks_df["text"].tolist(),
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
)

embeddings = np.asarray(
    embeddings,
    dtype=np.float32,
)

print(
    "Embeddings:",
    embeddings.shape,
)


In [ ]:
client = chromadb.PersistentClient(
    path=str(
        VECTOR_DB_DIR
    )
)

try:
    client.delete_collection(
        COLLECTION_NAME
    )
except Exception:
    pass

collection = client.create_collection(
    name=COLLECTION_NAME,
    metadata={
        "description": (
            "RAG v2 normativa VUT Málaga"
        ),
        "hnsw:space": "cosine",
    },
)

metadatas = [
    {
        "source_file": row["source_file"],
        "short_title": row["short_title"],
        "document_type": row["document_type"],
        "administration": row["administration"],
        "scope": row["scope"],
        "page": int(row["page"]),
        "chunk_index": int(
            row["chunk_index"]
        ),
    }
    for _, row in chunks_df.iterrows()
]

collection.add(
    ids=chunks_df[
        "chunk_id"
    ].tolist(),
    documents=chunks_df[
        "text"
    ].tolist(),
    metadatas=metadatas,
    embeddings=embeddings.tolist(),
)

print(
    "Chunks indexados:",
    collection.count(),
)


# Fase 4A — Retrieval v2

## 6. Recuperación amplia + deduplicación semántica + MMR

La nueva estrategia es:

1. recuperar 15 candidatos semánticos;
2. aplicar un bonus pequeño si la pregunta cita explícitamente un documento;
3. descartar candidatos por debajo de un umbral mínimo;
4. eliminar fragmentos casi duplicados mediante similitud coseno;
5. seleccionar el contexto final mediante **MMR**;
6. limitar a un máximo de 2 chunks por documento.

MMR equilibra:

- relevancia con la pregunta;
- diversidad respecto a chunks ya seleccionados.

Esto busca evitar que tres preguntas distintas terminen recibiendo exactamente los mismos fragmentos.


In [ ]:
DOCUMENT_ALIASES = {
    "Decreto 31/2024": [
        "decreto 31/2024",
        "decreto 31 2024",
        "decreto 31",
    ],
    "Decreto 28/2016": [
        "decreto 28/2016",
        "decreto 28 2016",
        "decreto 28",
    ],
    "Instrucción 1/2024": [
        "instrucción 1/2024",
        "instruccion 1/2024",
        "instrucción 1 2024",
        "instruccion 1 2024",
    ],
    "Resolución sobre Instrucción 1/2024": [
        "resolución",
        "resolucion",
    ],
    "Informe jurídico sobre Instrucción 1/2024": [
        "informe jurídico",
        "informe juridico",
    ],
    "Memoria modificación PGOU VUT": [
        "memoria",
        "pgou",
    ],
    "Resumen Ejecutivo PGOU VUT": [
        "resumen ejecutivo",
    ],
    "Acuerdo Pleno aprobación definitiva": [
        "acuerdo pleno",
        "acuerdo del pleno",
        "aprobación definitiva",
        "aprobacion definitiva",
    ],
    "Informe de Impacto de la Vivienda Turística": [
        "informe de impacto",
        "impacto de la vivienda turística",
        "impacto de la vivienda turistica",
    ],
}


def normalize_text(text):
    text = unicodedata.normalize(
        "NFKD",
        str(text),
    )

    text = "".join(
        char
        for char in text
        if not unicodedata.combining(
            char
        )
    )

    text = text.lower()

    text = re.sub(
        r"\s+",
        " ",
        text,
    )

    return text.strip()


def detect_explicit_documents(query):
    query_normalized = normalize_text(
        query
    )

    detected = []

    for title, aliases in (
        DOCUMENT_ALIASES.items()
    ):
        normalized_aliases = [
            normalize_text(alias)
            for alias in aliases
        ]

        if any(
            alias in query_normalized
            for alias in normalized_aliases
        ):
            detected.append(title)

    return detected


def cosine_similarity_matrix(matrix):
    matrix = np.asarray(
        matrix,
        dtype=np.float32,
    )

    norms = np.linalg.norm(
        matrix,
        axis=1,
        keepdims=True,
    )

    norms = np.clip(
        norms,
        1e-12,
        None,
    )

    normalized = matrix / norms

    return normalized @ normalized.T


In [ ]:
CHANGE_TERMS = [
    "modifica",
    "modificación",
    "modificacion",
    "queda modificado",
    "queda modificada",
    "se incorpora",
    "se incorporan",
    "se añade",
    "se anade",
    "nuevos requisitos",
    "nuevo requisito",
    "se sustituye",
    "se refuerza",
]

RESIDENTIAL_IMPACT_TERMS = [
    "impacto",
    "vivienda residencial",
    "acceso a la vivienda",
    "alquiler",
    "precios de alquiler",
    "precio de la vivienda",
    "hogares",
    "población",
    "poblacion",
    "viviendas principales",
    "viviendas vacías",
    "viviendas vacias",
    "presión turística residencial",
    "presion turistica residencial",
    "tensionando",
    "pierden",
    "desplazamiento",
]


def is_change_query(query):
    query_norm = normalize_text(query)

    change_query_terms = [
        "que cambios",
        "que cambia",
        "que modifica",
        "que modificaciones",
        "que introduce",
        "que incorpora",
    ]

    return any(
        term in query_norm
        for term in change_query_terms
    )


def is_residential_impact_query(query):
    query_norm = normalize_text(query)

    return (
        (
            "impacto" in query_norm
            or "efecto" in query_norm
            or "consecuencia" in query_norm
        )
        and (
            "vivienda" in query_norm
            or "residencial" in query_norm
            or "alquiler" in query_norm
        )
    )


def lexical_change_bonus(text):
    text_norm = normalize_text(text)

    hits = sum(
        1
        for term in CHANGE_TERMS
        if normalize_text(term) in text_norm
    )

    return min(
        0.12,
        0.03 * hits,
    )


def lexical_residential_bonus(text):
    text_norm = normalize_text(text)

    hits = sum(
        1
        for term in RESIDENTIAL_IMPACT_TERMS
        if normalize_text(term) in text_norm
    )

    # El bonus es deliberadamente moderado:
    # ayuda a distinguir impacto residencial de regulación,
    # sin sustituir la similitud semántica.
    return min(
        0.18,
        0.025 * hits,
    )


def expand_query_for_retrieval(query):
    """
    Expansión muy controlada solo para intenciones donde la consulta
    original es demasiado general para recuperar evidencia concreta.
    """
    if is_residential_impact_query(query):
        return (
            f"{query} "
            "impacto sobre vivienda residencial acceso a la vivienda "
            "precios de alquiler hogares población viviendas principales "
            "viviendas vacías presión turística residencial"
        )

    return query


def retrieve_candidates(
    query,
    candidate_pool=CANDIDATE_POOL,
    document_bonus=0.05,
):
    retrieval_query = expand_query_for_retrieval(
        query
    )

    # En preguntas de impacto residencial ampliamos el pool
    # porque los mejores fragmentos pueden no estar entre los
    # primeros resultados de una consulta muy genérica.
    if is_residential_impact_query(query):
        candidate_pool = max(
            candidate_pool,
            30,
        )

    query_embedding = embedding_model.encode(
        [retrieval_query],
        normalize_embeddings=True,
    )

    n_candidates = min(
        candidate_pool,
        collection.count(),
    )

    results = collection.query(
        query_embeddings=query_embedding.tolist(),
        n_results=n_candidates,
        include=[
            "documents",
            "metadatas",
            "distances",
            "embeddings",
        ],
    )

    explicit_documents = detect_explicit_documents(
        query
    )

    rows = []

    for i in range(
        len(results["documents"][0])
    ):
        metadata = results["metadatas"][0][i]
        text = results["documents"][0][i]

        distance = float(
            results["distances"][0][i]
        )

        semantic_similarity = 1.0 - distance

        explicit_match = (
            metadata["short_title"]
            in explicit_documents
        )

        residential_bonus = (
            lexical_residential_bonus(text)
            if is_residential_impact_query(query)
            else 0.0
        )

        effective_relevance = (
            semantic_similarity
            + (
                document_bonus
                if explicit_match
                else 0.0
            )
            + residential_bonus
        )

        rows.append(
            {
                "candidate_rank": i + 1,
                "documento": metadata[
                    "short_title"
                ],
                "tipo": metadata[
                    "document_type"
                ],
                "pagina": int(
                    metadata["page"]
                ),
                "chunk_index": int(
                    metadata.get(
                        "chunk_index",
                        0,
                    )
                ),
                "texto": text,
                "distance": distance,
                "semantic_similarity": (
                    semantic_similarity
                ),
                "lexical_bonus": residential_bonus,
                "explicit_document_match": (
                    explicit_match
                ),
                "effective_relevance": (
                    effective_relevance
                ),
                "embedding": np.asarray(
                    results["embeddings"][0][i],
                    dtype=np.float32,
                ),
            }
        )

    return pd.DataFrame(rows)


def retrieve_explicit_document(
    query,
    document_title,
    n_results=12,
):
    """
    Recuperación directa dentro del documento citado.
    Para preguntas de cambios/modificaciones añade un bonus léxico.
    """
    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True,
    )[0]

    results = collection.query(
        query_embeddings=[
            query_embedding.tolist()
        ],
        n_results=n_results,
        where={
            "short_title": document_title
        },
        include=[
            "documents",
            "metadatas",
            "distances",
            "embeddings",
        ],
    )

    rows = []
    change_query = is_change_query(query)

    for i in range(
        len(results["documents"][0])
    ):
        metadata = results["metadatas"][0][i]
        text = results["documents"][0][i]

        distance = float(
            results["distances"][0][i]
        )

        semantic_similarity = 1.0 - distance

        lexical_bonus = (
            lexical_change_bonus(text)
            if change_query
            else 0.0
        )

        rows.append(
            {
                "candidate_rank": i + 1,
                "documento": metadata[
                    "short_title"
                ],
                "tipo": metadata[
                    "document_type"
                ],
                "pagina": int(
                    metadata["page"]
                ),
                "chunk_index": int(
                    metadata.get(
                        "chunk_index",
                        0,
                    )
                ),
                "texto": text,
                "distance": distance,
                "semantic_similarity": (
                    semantic_similarity
                ),
                "lexical_bonus": lexical_bonus,
                "explicit_document_match": True,
                "effective_relevance": (
                    semantic_similarity
                    + 0.15
                    + lexical_bonus
                ),
                "embedding": np.asarray(
                    results["embeddings"][0][i],
                    dtype=np.float32,
                ),
            }
        )

    return pd.DataFrame(rows)


def get_neighbor_chunk(
    document_title,
    page,
    chunk_index,
    direction=1,
):
    document_chunks = chunks_df.loc[
        chunks_df["short_title"]
        == document_title
    ].copy()

    document_chunks = (
        document_chunks
        .sort_values(
            [
                "page",
                "chunk_index",
            ]
        )
        .reset_index(drop=True)
    )

    current_matches = document_chunks.index[
        (
            document_chunks["page"]
            == page
        )
        & (
            document_chunks["chunk_index"]
            == chunk_index
        )
    ].tolist()

    if not current_matches:
        return None

    neighbor_position = (
        current_matches[0]
        + direction
    )

    if (
        neighbor_position < 0
        or neighbor_position
        >= len(document_chunks)
    ):
        return None

    return document_chunks.iloc[
        neighbor_position
    ]


def remove_near_duplicates(
    candidates,
    threshold=NEAR_DUPLICATE_THRESHOLD,
):
    if candidates.empty:
        return candidates.copy()

    candidates = (
        candidates
        .sort_values(
            [
                "effective_relevance",
                "semantic_similarity",
            ],
            ascending=False,
        )
        .reset_index(drop=True)
    )

    kept_rows = []
    kept_embeddings = []

    for _, row in candidates.iterrows():
        current_embedding = row[
            "embedding"
        ]

        if not kept_embeddings:
            kept_rows.append(
                row.to_dict()
            )
            kept_embeddings.append(
                current_embedding
            )
            continue

        similarities = [
            float(
                np.dot(
                    current_embedding,
                    previous_embedding,
                )
            )
            for previous_embedding
            in kept_embeddings
        ]

        if max(similarities) < threshold:
            kept_rows.append(
                row.to_dict()
            )
            kept_embeddings.append(
                current_embedding
            )

    return pd.DataFrame(
        kept_rows
    )


def mmr_select(
    candidates,
    top_k=FINAL_TOP_K,
    lambda_mult=MMR_LAMBDA,
    max_per_document=MAX_PER_DOCUMENT,
):
    if candidates.empty:
        return candidates.copy()

    candidates = (
        candidates
        .sort_values(
            "effective_relevance",
            ascending=False,
        )
        .reset_index(drop=True)
    )

    selected_indices = []
    document_counts = {}

    while (
        len(selected_indices) < top_k
        and len(selected_indices)
        < len(candidates)
    ):
        best_index = None
        best_score = -np.inf

        for idx, row in candidates.iterrows():
            if idx in selected_indices:
                continue

            document = row["documento"]

            if (
                document_counts.get(
                    document,
                    0,
                )
                >= max_per_document
            ):
                continue

            relevance = float(
                row["effective_relevance"]
            )

            if not selected_indices:
                diversity_penalty = 0.0
            else:
                current_embedding = row[
                    "embedding"
                ]

                diversity_penalty = max(
                    float(
                        np.dot(
                            current_embedding,
                            candidates.loc[
                                selected_idx,
                                "embedding",
                            ],
                        )
                    )
                    for selected_idx
                    in selected_indices
                )

            mmr_score = (
                lambda_mult
                * relevance
                - (
                    1.0 - lambda_mult
                )
                * diversity_penalty
            )

            if mmr_score > best_score:
                best_score = mmr_score
                best_index = idx

        if best_index is None:
            break

        selected_indices.append(
            best_index
        )

        selected_document = candidates.loc[
            best_index,
            "documento",
        ]

        document_counts[
            selected_document
        ] = (
            document_counts.get(
                selected_document,
                0,
            )
            + 1
        )

        candidates.loc[
            best_index,
            "mmr_score",
        ] = best_score

    selected = (
        candidates
        .loc[selected_indices]
        .copy()
        .reset_index(drop=True)
    )

    selected.insert(
        0,
        "rank",
        np.arange(
            1,
            len(selected) + 1,
        ),
    )

    return selected


def merge_open_change_continuations(
    selected,
    query,
    document_title,
):
    """
    Si un fragmento seleccionado anuncia que una norma 'queda modificada
    en los siguientes términos', concatena el chunk inmediatamente posterior
    dentro del mismo resultado.

    Así no se desperdicia uno de los 4 puestos con un mero encabezado y,
    al mismo tiempo, se conserva la continuidad jurídica entre páginas/chunks.
    """
    if (
        selected.empty
        or not is_change_query(query)
    ):
        return selected.copy()

    open_ending_terms = [
        "queda modificado",
        "queda modificada",
        "en los siguientes terminos",
        "se modifica",
    ]

    merged = selected.copy()

    for idx, row in merged.iterrows():
        text_norm = normalize_text(
            row["texto"]
        )

        if not any(
            normalize_text(term)
            in text_norm
            for term in open_ending_terms
        ):
            continue

        neighbor = get_neighbor_chunk(
            document_title=document_title,
            page=int(row["pagina"]),
            chunk_index=int(
                row["chunk_index"]
            ),
            direction=1,
        )

        if neighbor is None:
            continue

        neighbor_text = str(
            neighbor["text"]
        ).strip()

        if not neighbor_text:
            continue

        neighbor_page = int(
            neighbor["page"]
        )

        merged.at[
            idx,
            "texto",
        ] = (
            str(row["texto"]).rstrip()
            + "\n\n"
            + (
                f"[Continuación inmediata, "
                f"página {neighbor_page}]\n"
            )
            + neighbor_text
        )

        merged.at[
            idx,
            "continuation_page",
        ] = neighbor_page

    return merged


def retrieve_documents(
    query,
    n_results=FINAL_TOP_K,
    min_similarity=MIN_SIMILARITY,
):
    """
    Retrieval híbrido v2.2.

    DOCUMENTO EXPLÍCITO:
    - consulta filtrada al documento citado;
    - bonus léxico para preguntas de cambios;
    - continuidad jurídica: un encabezado de modificación se fusiona
      con su chunk inmediatamente posterior.

    PREGUNTA GENERAL:
    - búsqueda semántica;
    - expansión controlada para impacto residencial;
    - candidate pool ampliado en esa intención;
    - bonus léxico temático;
    - deduplicación + MMR + diversidad documental.
    """
    explicit_documents = detect_explicit_documents(
        query
    )

    # =====================================================
    # A. Documento citado explícitamente
    # =====================================================
    if explicit_documents:
        primary_document = explicit_documents[0]

        candidates = retrieve_explicit_document(
            query=query,
            document_title=primary_document,
            n_results=12,
        )

        if not candidates.empty:
            candidates = candidates.loc[
                candidates[
                    "semantic_similarity"
                ]
                >= min_similarity
            ].copy()

        if candidates.empty:
            # Fallback general si el filtro documental no recupera
            # evidencia suficiente.
            candidates = retrieve_candidates(
                query
            )

            if candidates.empty:
                return candidates

            candidates = candidates.loc[
                candidates[
                    "semantic_similarity"
                ]
                >= min_similarity
            ].copy()

            if candidates.empty:
                return candidates

            selected = mmr_select(
                remove_near_duplicates(
                    candidates
                ),
                top_k=n_results,
            )

        else:
            deduplicated = remove_near_duplicates(
                candidates
            )

            selected = mmr_select(
                deduplicated,
                top_k=n_results,
                max_per_document=n_results,
            )

            selected = (
                merge_open_change_continuations(
                    selected=selected,
                    query=query,
                    document_title=primary_document,
                )
            )

    # =====================================================
    # B. Pregunta general
    # =====================================================
    else:
        candidates = retrieve_candidates(
            query
        )

        if candidates.empty:
            return candidates

        candidates = candidates.loc[
            candidates[
                "semantic_similarity"
            ]
            >= min_similarity
        ].copy()

        if candidates.empty:
            return candidates

        deduplicated = remove_near_duplicates(
            candidates
        )

        selected = mmr_select(
            deduplicated,
            top_k=n_results,
            max_per_document=(
                MAX_PER_DOCUMENT
            ),
        )

    if selected.empty:
        return selected

    selected = (
        selected
        .drop_duplicates(
            subset=[
                "documento",
                "pagina",
                "chunk_index",
            ],
            keep="first",
        )
        .reset_index(drop=True)
        .drop(
            columns=[
                "embedding",
            ],
            errors="ignore",
        )
    )

    selected["rank"] = np.arange(
        1,
        len(selected) + 1,
    )

    columns = [
        "rank"
    ] + [
        column
        for column in selected.columns
        if column != "rank"
    ]

    return selected[columns]

## 7. Diagnóstico del retrieval antes de usar un LLM

Esta fase es importante: primero se comprueba si preguntas distintas reciben realmente contextos distintos.

La tabla incluye:

- similitud semántica;
- documento y página;
- ranking final;
- coincidencia explícita de documento.

Todavía **no se llama a Ollama**.


In [ ]:
retrieval_test_questions = [
    "¿Qué requisitos debe cumplir una vivienda de uso turístico?",
    (
        "¿Qué ocurre cuando un barrio alcanza o supera "
        "el 8% de viviendas de uso turístico?"
    ),
    (
        "¿Qué impacto tienen las viviendas de uso turístico "
        "sobre la vivienda residencial en Málaga?"
    ),
    (
        "¿Qué establece la Instrucción 1/2024 "
        "sobre las viviendas de uso turístico?"
    ),
    (
        "¿Qué cambios introduce el Decreto 31/2024 "
        "en la regulación de las viviendas de uso turístico?"
    ),
    (
        "¿Qué regula el Decreto 28/2016 "
        "sobre viviendas de uso turístico?"
    ),
    (
        "¿Puede el Ayuntamiento de Málaga limitar "
        "la implantación de nuevas VUT?"
    ),
    (
        "¿Cuál es la sanción económica exacta "
        "por alquilar una VUT sin licencia en Málaga?"
    ),
]

retrieval_rows = []

for question in retrieval_test_questions:
    results = retrieve_documents(
        question
    )

    for _, row in results.iterrows():
        retrieval_rows.append(
            {
                "pregunta": question,
                **row.to_dict(),
            }
        )

retrieval_audit_df = pd.DataFrame(
    retrieval_rows
)

audit_columns = [
    "pregunta",
    "rank",
    "documento",
    "pagina",
    "chunk_index",
    "semantic_similarity",
    "explicit_document_match",
]

if "lexical_bonus" in retrieval_audit_df.columns:
    audit_columns.append(
        "lexical_bonus"
    )

display(
    retrieval_audit_df[
        audit_columns
    ]
)

RETRIEVAL_AUDIT_PATH = (
    REPORTS_RAG_DIR
    / "retrieval_v2_1_audit.csv"
)

retrieval_audit_df.to_csv(
    RETRIEVAL_AUDIT_PATH,
    index=False,
    encoding="utf-8-sig",
)

print(
    "Auditoría guardada:",
    RETRIEVAL_AUDIT_PATH,
)

### 7.1. Medida simple de diversidad

Se calcula cuántos documentos distintos aparecen entre los resultados finales de cada pregunta.

No sustituye una evaluación cualitativa, pero permite detectar rápidamente si el retrieval está colapsando siempre en la misma fuente.


In [ ]:
retrieval_diversity_df = (
    retrieval_audit_df
    .groupby(
        "pregunta",
        as_index=False,
    )
    .agg(
        n_chunks=("rank", "count"),
        n_documentos=(
            "documento",
            "nunique",
        ),
        documentos=(
            "documento",
            lambda values: " | ".join(
                pd.unique(values)
            ),
        ),
        similitud_media=(
            "semantic_similarity",
            "mean",
        ),
    )
)

retrieval_diversity_df[
    "similitud_media"
] = (
    retrieval_diversity_df[
        "similitud_media"
    ]
    .round(3)
)

display(
    retrieval_diversity_df
)


# Fase 4B — Generación

## 8. Construcción del contexto y abstención

Antes de llamar al LLM se comprueba que exista evidencia recuperada por encima del umbral.

Si no hay chunks suficientes, la función devuelve una abstención determinista y **no llama al modelo**.


In [ ]:
def format_rag_context(
    retrieved_df
):
    context_parts = []
    source_map = {}

    for i, row in (
        retrieved_df.iterrows()
    ):
        source_id = f"S{i + 1}"

        source_map[
            source_id
        ] = {
            "documento": row[
                "documento"
            ],
            "pagina": int(
                row["pagina"]
            ),
        }

        similarity = row.get(
            "semantic_similarity",
            np.nan,
        )

        similarity_text = (
            f"{float(similarity):.3f}"
            if pd.notna(similarity)
            else "contexto_vecino"
        )

        context_parts.append(
            (
                f"[{source_id}]\n"
                f"FUENTE: {row['documento']}\n"
                f"TIPO: {row.get('tipo', '')}\n"
                f"PÁGINA: {int(row['pagina'])}\n"
                f"SIMILITUD: {similarity_text}\n\n"
                f"{row['texto']}"
            )
        )

    return (
        "\n\n---\n\n".join(
            context_parts
        ),
        source_map,
    )


def replace_source_ids(
    answer,
    source_map,
):
    def replacement(match):
        source_id = match.group(1)

        if source_id not in source_map:
            return match.group(0)

        source = source_map[
            source_id
        ]

        return (
            f"[{source['documento']}, "
            f"p. {source['pagina']}]"
        )

    return re.sub(
        r"\[(S\d+)\]",
        replacement,
        answer,
    )

## 9. Prompt v3

El prompt obliga al LLM a revisar todos los fragmentos, prioriza el documento
citado explícitamente y reduce dos errores observados durante las pruebas:

- responder solo con el primer chunk;
- atribuir a una norma afirmaciones procedentes de una fuente complementaria.

La trazabilidad principal sigue estando garantizada por la tabla de fuentes
recuperadas; las citas generadas por el LLM son un apoyo adicional.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

rag_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
Eres un asistente documental especializado en viviendas de uso turístico
(VUT) en Málaga y Andalucía.

Debes responder ÚNICAMENTE con la información incluida en el CONTEXTO.

PROCEDIMIENTO OBLIGATORIO:

Antes de redactar la respuesta:
- revisa TODOS los fragmentos recuperados;
- identifica qué aporta cada fragmento a la pregunta;
- descarta únicamente los fragmentos que no sean relevantes;
- no te limites al primer fragmento si otros aportan información
  complementaria necesaria.

REGLAS:

1. Responde exactamente a la pregunta formulada.

2. Si la pregunta menciona expresamente un documento
   (por ejemplo, "¿Qué establece la Instrucción 1/2024?"),
   sintetiza las principales ideas relevantes recuperadas DE ESE
   DOCUMENTO. No atribuyas al documento principal afirmaciones que
   procedan de otra fuente.

3. Cuando varios fragmentos del documento solicitado aporten
   aspectos diferentes y relevantes, intégralos en la respuesta.

4. No añadas información que no aparezca en el contexto.

5. No utilices conocimiento general ni conocimiento previo del modelo.

6. Distingue, cuando sea necesario, entre:
   - normativa autonómica;
   - normativa o instrumentos municipales;
   - planeamiento urbanístico;
   - informes técnicos o jurídicos.

7. No confundas una norma citada dentro de un documento con
   el propio contenido o finalidad del documento consultado.

8. Si varias fuentes repiten la misma idea, exprésala una sola vez.

9. Si la evidencia recuperada no permite responder con seguridad,
   responde exactamente:
   "La documentación recuperada no aporta evidencia suficiente para
   responder con precisión a esta pregunta."

10. Cada afirmación importante debe indicar la fuente que la respalda
    mediante [S1], [S2], [S3] o [S4].

11. Solo puedes utilizar identificadores presentes en el CONTEXTO.

12. No inventes fuentes, artículos, cifras ni consecuencias.

13. Para preguntas amplias, organiza la respuesta en varios puntos
    breves cuando existan varias ideas diferentes.

14. Evita repetir una conclusión en un párrafo final si ya ha quedado
    explicada en la respuesta.

15. La respuesta debe ser concisa, pero suficientemente completa
    para responder a toda la pregunta.

16. La respuesta es informativa y no sustituye asesoramiento jurídico.
            """,
        ),
        (
            "human",
            """
PREGUNTA:
{question}

CONTEXTO:
{context}

Analiza todos los fragmentos relevantes y responde únicamente
a la pregunta formulada.
            """,
        ),
    ]
)

print("Prompt v3 preparado")

## 10. Modelo generativo final y función RAG

Tras la comparación experimental entre `llama3.2:3b` y `mistral:7b`,
se selecciona `mistral:7b` como modelo final por su mayor calidad global
en precisión, cobertura y fidelidad documental.

La función `ask_rag()` ejecuta el flujo completo:

1. retrieval;
2. construcción del contexto;
3. generación con el LLM;
4. sustitución de identificadores de fuente;
5. devolución de respuesta, fuentes y tiempo de generación.

In [ ]:
from langchain_ollama import ChatOllama

llm_cache = {}


def get_llm(model_name):
    if model_name not in llm_cache:
        llm_cache[
            model_name
        ] = ChatOllama(
            model=model_name,
            temperature=0,
        )

    return llm_cache[
        model_name
    ]


def ask_rag(
    question,
    model_name=FINAL_LLM,
    n_results=FINAL_TOP_K,
):
    retrieved = retrieve_documents(
        question,
        n_results=n_results,
    )

    if retrieved.empty:
        return {
            "question": question,
            "model": model_name,
            "answer": (
                "La documentación recuperada no aporta "
                "evidencia suficiente para responder "
                "con precisión a esta pregunta."
            ),
            "raw_answer": None,
            "sources": retrieved,
            "generation_seconds": 0.0,
            "abstained_before_llm": True,
        }

    context, source_map = (
        format_rag_context(
            retrieved
        )
    )

    llm = get_llm(
        model_name
    )

    chain = (
        rag_prompt
        | llm
    )

    start = time.perf_counter()

    response = chain.invoke(
        {
            "question": question,
            "context": context,
        }
    )

    elapsed = (
        time.perf_counter()
        - start
    )

    raw_answer = str(
        response.content
    ).strip()

    final_answer = (
        replace_source_ids(
            raw_answer,
            source_map,
        )
    )

    return {
        "question": question,
        "model": model_name,
        "answer": final_answer,
        "raw_answer": raw_answer,
        "sources": retrieved,
        "generation_seconds": elapsed,
        "abstained_before_llm": False,
    }


# Fase 4C — Validación final

## 11. Banco final de evaluación

Se utilizan 8 preguntas representativas que cubren:

- modificación normativa;
- aplicación de la Instrucción 1/2024;
- umbral municipal del 8%;
- impacto residencial;
- requisitos de las VUT;
- alcance del Decreto 28/2016;
- capacidad municipal de limitar nuevas VUT;
- una pregunta negativa diseñada para comprobar abstención.

In [ ]:
FINAL_EVALUATION_QUESTIONS = {
    "decreto31_cambios": (
        "¿Qué cambios introduce el Decreto 31/2024?"
    ),
    "instruccion1_2024": (
        "¿Qué establece la Instrucción 1/2024 "
        "sobre las viviendas de uso turístico?"
    ),
    "umbral_8_por_ciento": (
        "¿Qué ocurre cuando un barrio supera "
        "el 8% de viviendas de uso turístico?"
    ),
    "impacto_residencial": (
        "¿Qué impacto tienen las VUT "
        "sobre la vivienda residencial en Málaga?"
    ),
    "requisitos_vut": (
        "¿Qué requisitos debe cumplir "
        "una vivienda de uso turístico?"
    ),
    "decreto28_regulacion": (
        "¿Qué regula el Decreto 28/2016 "
        "sobre viviendas de uso turístico?"
    ),
    "limite_ayuntamiento": (
        "¿Puede el Ayuntamiento de Málaga limitar "
        "la implantación de nuevas VUT?"
    ),
    "sancion_exacta": (
        "¿Cuál es la sanción económica exacta "
        "por alquilar una VUT sin licencia en Málaga?"
    ),
}

evaluation_catalog = pd.DataFrame(
    {
        "clave": list(
            FINAL_EVALUATION_QUESTIONS.keys()
        ),
        "pregunta": list(
            FINAL_EVALUATION_QUESTIONS.values()
        ),
    }
)

display(
    evaluation_catalog
)

print(
    "Modelo final:",
    FINAL_LLM,
)

print(
    "Número de preguntas:",
    len(FINAL_EVALUATION_QUESTIONS),
)

## 12. Check final del retrieval

Esta comprobación no llama al LLM. Sirve para dejar registrada la calidad del
contexto recuperado antes de la evaluación generativa.

In [ ]:
retrieval_check_rows = []

for key, question in (
    FINAL_EVALUATION_QUESTIONS.items()
):
    retrieved = retrieve_documents(
        question,
        n_results=FINAL_TOP_K,
    )

    retrieval_check_rows.append(
        {
            "clave": key,
            "pregunta": question,
            "n_chunks": len(retrieved),
            "n_documentos": (
                retrieved["documento"].nunique()
                if not retrieved.empty
                else 0
            ),
            "documentos": (
                " | ".join(
                    retrieved[
                        "documento"
                    ].astype(str)
                )
                if not retrieved.empty
                else ""
            ),
            "sim_media": (
                retrieved[
                    "semantic_similarity"
                ].mean()
                if not retrieved.empty
                else np.nan
            ),
            "sim_min": (
                retrieved[
                    "semantic_similarity"
                ].min()
                if not retrieved.empty
                else np.nan
            ),
            "sim_max": (
                retrieved[
                    "semantic_similarity"
                ].max()
                if not retrieved.empty
                else np.nan
            ),
        }
    )

retrieval_check_df = pd.DataFrame(
    retrieval_check_rows
)

display(
    retrieval_check_df
)

RETRIEVAL_FINAL_PATH = (
    REPORTS_RAG_DIR
    / "retrieval_v2_2_final_check.csv"
)

retrieval_check_df.to_csv(
    RETRIEVAL_FINAL_PATH,
    index=False,
    encoding="utf-8-sig",
)

print(
    "Check de retrieval guardado en:",
    RETRIEVAL_FINAL_PATH,
)

## 13. Evaluación generativa final con Mistral 7B

Esta es la única sección lenta del notebook. Ejecuta las 8 preguntas de forma
secuencial con el modelo final y guarda automáticamente las respuestas y las
fuentes utilizadas.

En el hardware de desarrollo puede tardar aproximadamente entre 40 y 70 minutos.

In [ ]:
final_evaluation_rows = []
final_sources_rows = []

total_questions = len(
    FINAL_EVALUATION_QUESTIONS
)

for i, (
    key,
    question,
) in enumerate(
    FINAL_EVALUATION_QUESTIONS.items(),
    start=1,
):
    print(
        "\n",
        "=" * 100,
    )
    print(
        f"[{i}/{total_questions}] {key}"
    )
    print(
        question
    )

    result = ask_rag(
        question,
        model_name=FINAL_LLM,
    )

    print(
        "\nRESPUESTA:\n"
    )
    print(
        result["answer"]
    )

    print(
        "\nTIEMPO:",
        round(
            result[
                "generation_seconds"
            ],
            1,
        ),
        "s",
    )

    final_evaluation_rows.append(
        {
            "clave": key,
            "pregunta": question,
            "modelo": result[
                "model"
            ],
            "tiempo_s": result[
                "generation_seconds"
            ],
            "respuesta": result[
                "answer"
            ],
            "abstencion_previa_llm": result[
                "abstained_before_llm"
            ],
            "n_fuentes": len(
                result["sources"]
            ),
            "documentos_contexto": (
                " | ".join(
                    pd.unique(
                        result[
                            "sources"
                        ][
                            "documento"
                        ]
                    )
                )
                if not result[
                    "sources"
                ].empty
                else ""
            ),
        }
    )

    if not result["sources"].empty:
        for _, source in (
            result["sources"].iterrows()
        ):
            final_sources_rows.append(
                {
                    "clave": key,
                    "pregunta": question,
                    "modelo": result[
                        "model"
                    ],
                    "rank": int(
                        source["rank"]
                    ),
                    "documento": source[
                        "documento"
                    ],
                    "pagina": int(
                        source["pagina"]
                    ),
                    "chunk_index": int(
                        source[
                            "chunk_index"
                        ]
                    ),
                    "semantic_similarity": (
                        source[
                            "semantic_similarity"
                        ]
                    ),
                    "texto": source[
                        "texto"
                    ],
                }
            )

final_evaluation_df = pd.DataFrame(
    final_evaluation_rows
)

final_sources_df = pd.DataFrame(
    final_sources_rows
)

display(
    final_evaluation_df
)

FINAL_EVAL_PATH = (
    REPORTS_RAG_DIR
    / "evaluacion_rag_v2_2_mistral_final.csv"
)

FINAL_SOURCES_PATH = (
    REPORTS_RAG_DIR
    / "evaluacion_rag_v2_2_mistral_fuentes.csv"
)

final_evaluation_df.to_csv(
    FINAL_EVAL_PATH,
    index=False,
    encoding="utf-8-sig",
)

final_sources_df.to_csv(
    FINAL_SOURCES_PATH,
    index=False,
    encoding="utf-8-sig",
)

print(
    "\nEvaluación final guardada en:",
    FINAL_EVAL_PATH,
)

print(
    "Fuentes de la evaluación guardadas en:",
    FINAL_SOURCES_PATH,
)

## 14. Resumen final de la Fase 4

La versión final del motor conversacional queda formada por:

- corpus documental local;
- chunking estructurado;
- embeddings multilingües;
- ChromaDB;
- retrieval híbrido v2.2;
- prompt v3;
- `mistral:7b` como LLM final;
- trazabilidad mediante fuentes recuperadas;
- mecanismo de abstención cuando la documentación no aporta evidencia suficiente.

Los CSV generados en `reports/rag_v2` dejan registrada la validación del retrieval
y la evaluación final del sistema antes de integrarlo en FastAPI y Streamlit.